# DDR Cell-State Point Clouds from MultiFoci

This notebook builds **one point cloud per cell/nucleus** from `MultiFoci_combined.parquet` and prepares a supervised dataset for **cell-state classification**.

Pipeline:
1. Load classified cells + MultiFoci tables
2. Join foci rows to parent cell labels
3. Build per-cell point clouds (variable number of foci per cell)
4. Save reusable artifacts
5. Run a quick baseline classifier sanity check

In [1]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import re
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, balanced_accuracy_score, confusion_matrix

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

## 1) Configuration

In [2]:
ROOT = Path('/nfs/roberts/pi/pi_sk2433/shared/JohnLock_2026_DDR')
CELLS_FILE = ROOT / 'cells_classified-003.parquet'
MULTI_FILE = ROOT / 'MultiFoci_combined.parquet'

OUT_DIR = Path('DDR/pointcloud_outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Choose one label target for supervised classification.
# Options seen during pre-analysis: 'Cell_State_Thr', 'Cell_State_Score', 'CC_cluster'
TARGET_LABEL = 'Cell_State_Thr'

# Point-cloud construction controls
MIN_POINTS_PER_CELL = 1
MAX_POINTS_PER_CELL = 256  # subsample if a cell has too many foci

print('Cells file exists:', CELLS_FILE.exists())
print('Multi file exists:', MULTI_FILE.exists())
print('Output directory:', OUT_DIR.resolve())

Cells file exists: True
Multi file exists: True
Output directory: /nfs/roberts/project/pi_sk2433/sv496/HiPoNet/DDR/DDR/pointcloud_outputs


## 2) Use All Numeric MultiFoci Columns with No Missing Values

This section starts from **all columns** in `MultiFoci_combined.parquet` (except the join keys), keeps the numeric ones, and then selects every numeric feature with zero missingness in the joined data.

In [3]:
import pyarrow as pa
import pyarrow.types as pat

cells_cols_all = pq.ParquetFile(CELLS_FILE).schema.names
pf_multi = pq.ParquetFile(MULTI_FILE)
multi_cols_all = pf_multi.schema.names
multi_schema = pf_multi.schema_arrow

required_cells = ['Nuclei_ImageNumber', 'Nuclei_ObjectNumber', TARGET_LABEL]
missing_cells = [c for c in required_cells if c not in cells_cols_all]
if missing_cells:
    raise ValueError(f'Missing required columns in cells file: {missing_cells}')

required_multi_keys = ['ImageNumber', 'ObjectNumber']
missing_multi_keys = [c for c in required_multi_keys if c not in multi_cols_all]
if missing_multi_keys:
    raise ValueError(f'Missing required key columns in MultiFoci: {missing_multi_keys}')

# Use all MultiFoci columns except join keys as initial candidates.
point_feature_candidates = [c for c in multi_cols_all if c not in required_multi_keys]

# Keep only numeric columns based on parquet schema types.
def is_numeric_arrow_type(t: pa.DataType) -> bool:
    return (
        pat.is_integer(t)
        or pat.is_floating(t)
        or pat.is_decimal(t)
    )

numeric_candidate_cols = [
    f.name
    for f in multi_schema
    if f.name in point_feature_candidates and is_numeric_arrow_type(f.type)
]

# Remove the target label from candidates if it exists in MultiFoci (safety).
numeric_candidate_cols = [c for c in numeric_candidate_cols if c != TARGET_LABEL]

cells = pd.read_parquet(CELLS_FILE, columns=required_cells).rename(
    columns={'Nuclei_ImageNumber': 'ImageNumber', 'Nuclei_ObjectNumber': 'ObjectNumber'}
)
multi = pd.read_parquet(MULTI_FILE, columns=required_multi_keys + numeric_candidate_cols)

print('Loaded cells shape:', cells.shape)
print('Loaded MultiFoci shape:', multi.shape)
print('Total MultiFoci columns:', len(multi_cols_all))
print('All non-key candidates:', len(point_feature_candidates))
print('Numeric candidate feature columns:', len(numeric_candidate_cols))
print('First 30 numeric candidates:', numeric_candidate_cols[:30])

Loaded cells shape: (27039, 3)
Loaded MultiFoci shape: (114484, 2473)
Total MultiFoci columns: 2714
All non-key candidates: 2712
Numeric candidate feature columns: 2471
First 30 numeric candidates: ['Metadata_Chan_ID', 'Metadata_Cycle_ID', 'Metadata_ExpDate', 'Metadata_Field', 'Metadata_FileLocation', 'Metadata_Flour', 'Metadata_Frame', 'Metadata_Marker', 'Metadata_Registered', 'Metadata_Series', 'AreaShape_Area', 'AreaShape_BoundingBoxArea', 'AreaShape_BoundingBoxMaximum_X', 'AreaShape_BoundingBoxMaximum_Y', 'AreaShape_BoundingBoxMinimum_X', 'AreaShape_BoundingBoxMinimum_Y', 'AreaShape_Center_X', 'AreaShape_Center_Y', 'AreaShape_Compactness', 'AreaShape_ConvexArea', 'AreaShape_Eccentricity', 'AreaShape_EquivalentDiameter', 'AreaShape_EulerNumber', 'AreaShape_Extent', 'AreaShape_FormFactor', 'AreaShape_MajorAxisLength', 'AreaShape_MaxFeretDiameter', 'AreaShape_MaximumRadius', 'AreaShape_MeanRadius', 'AreaShape_MedianRadius']


## 3) Join Foci to Cell Labels + Filter Features by Missingness

This step joins foci rows to labeled cells and keeps only point-feature columns that pass the configured missingness threshold.

In [4]:
# Drop rows with missing label early
cells = cells.dropna(subset=[TARGET_LABEL]).copy()

# Inner join keeps only foci that can be mapped to a labeled nucleus
joined = multi.merge(cells, on=['ImageNumber', 'ObjectNumber'], how='inner')

# Keep every numeric feature with zero missingness in the joined data.
missing_frac = joined[numeric_candidate_cols].isna().mean().sort_values()
point_feature_cols = missing_frac[missing_frac == 0.0].index.tolist()

if not point_feature_cols:
    raise ValueError('No zero-missing numeric features found. Consider checking the parquet join or missing columns.')

print('Joined rows:', len(joined))
print('Unique labeled cells:', joined[['ImageNumber', 'ObjectNumber']].drop_duplicates().shape[0])
print('Cell-level label distribution (unique cells):')
print(cells[TARGET_LABEL].value_counts(dropna=False).to_string())
print('\nFoci-row label distribution (joined set):')
print(joined[TARGET_LABEL].value_counts(dropna=False).to_string())

print('\nSelected point feature count:', len(point_feature_cols))
print('Selected features:', point_feature_cols)

coord_cols = ['Location_Center_X', 'Location_Center_Y', 'Location_Center_Z']
coord_in_selected = [c for c in coord_cols if c in point_feature_cols]
coord_available = [c for c in coord_cols if c in numeric_candidate_cols]
print('\nCoordinate columns available in MultiFoci candidates:', coord_available)
print('Coordinate columns selected (zero missing):', coord_in_selected)

print('\nTop 20 lowest-missing candidate columns:')
print(missing_frac.head(20).to_string())

print('\nTop 20 highest-missing candidate columns:')
print(missing_frac.tail(20).to_string())

feature_audit_df = pd.DataFrame({'feature': missing_frac.index, 'missing_fraction': missing_frac.values})
feature_audit_df['selected'] = feature_audit_df['feature'].isin(point_feature_cols)
feature_audit_df.head(30)

Joined rows: 45337
Unique labeled cells: 21603
Cell-level label distribution (unique cells):
Cell_State_Thr
Intermediate            23397
Resisting Cell Death     1498
Stressed                  725
Proliferative             703
Early Apoptotic           439
Late Apoptotic            277

Foci-row label distribution (joined set):
Cell_State_Thr
Intermediate            38249
Resisting Cell Death     3135
Stressed                 1504
Proliferative            1486
Late Apoptotic            555
Early Apoptotic           408

Selected point feature count: 2460
Selected features: ['Intensity_UpperQuartileIntensity_Cycle01_DIC', 'Intensity_StdIntensity_Cycle22_DAPI', 'Intensity_StdIntensity_Cycle22_DIC', 'Intensity_StdIntensity_Cycle22_Vimentin', 'Intensity_StdIntensity_Cycle23_DAPI', 'Intensity_StdIntensity_Cycle23_DIC', 'Intensity_StdIntensity_Cycle23_LAMP1', 'Intensity_StdIntensity_Cycle23_Total protein', 'Intensity_StdIntensity_Cycle23_pCREB', 'Intensity_UpperQuartileIntensity_Cycle01_DAP

,feature,missing_fraction,selected
0,Intensity_UpperQuartileIntensity_Cycle01_DIC,0.0,True
1,Intensity_StdIntensity_Cycle22_DAPI,0.0,True
2,Intensity_StdIntensity_Cycle22_DIC,0.0,True
3,Intensity_StdIntensity_Cycle22_Vimentin,0.0,True
4,Intensity_StdIntensity_Cycle23_DAPI,0.0,True
5,Intensity_StdIntensity_Cycle23_DIC,0.0,True
6,Intensity_StdIntensity_Cycle23_LAMP1,0.0,True
7,Intensity_StdIntensity_Cycle23_Total protein,0.0,True
8,Intensity_StdIntensity_Cycle23_pCREB,0.0,True
9,Intensity_UpperQuartileIntensity_Cycle01_DAPI,0.0,True


## 4) Build Per-Cell Point Clouds (No-NaN Features)

Since selected columns already satisfy the missingness threshold, this section builds clouds without NaN imputation.

In [5]:
if not point_feature_cols:
    raise ValueError('No selected point-feature columns available.')

pc_df = joined[['ImageNumber', 'ObjectNumber', TARGET_LABEL] + point_feature_cols].copy()

# Safety check: there should be no NaNs in selected features under strict mode.
nan_total = int(pc_df[point_feature_cols].isna().sum().sum())
print('Total NaN values across selected features:', nan_total)
if nan_total > 0:
    raise ValueError(
        'Selected features still contain NaNs. '
        'Either tighten selection or explicitly impute based on your preference.'
    )

# Standardize point features globally across all foci rows
scaler = StandardScaler()
pc_df[point_feature_cols] = scaler.fit_transform(pc_df[point_feature_cols].values)

# Encode labels
label_encoder = LabelEncoder()
pc_df['_label_str'] = pc_df[TARGET_LABEL].astype(str)
pc_df['_label_id'] = label_encoder.fit_transform(pc_df['_label_str'])

print('Selected numeric point features:', point_feature_cols)
print('Num classes:', len(label_encoder.classes_))
print('Classes:', list(label_encoder.classes_))

Total NaN values across selected features: 0
Selected numeric point features: ['Intensity_UpperQuartileIntensity_Cycle01_DIC', 'Intensity_StdIntensity_Cycle22_DAPI', 'Intensity_StdIntensity_Cycle22_DIC', 'Intensity_StdIntensity_Cycle22_Vimentin', 'Intensity_StdIntensity_Cycle23_DAPI', 'Intensity_StdIntensity_Cycle23_DIC', 'Intensity_StdIntensity_Cycle23_LAMP1', 'Intensity_StdIntensity_Cycle23_Total protein', 'Intensity_StdIntensity_Cycle23_pCREB', 'Intensity_UpperQuartileIntensity_Cycle01_DAPI', 'Intensity_StdIntensity_Cycle22_Active YAP1', 'Intensity_UpperQuartileIntensity_Cycle01_Ku7080', 'Intensity_UpperQuartileIntensity_Cycle01_RAD51', 'Intensity_UpperQuartileIntensity_Cycle01_pH2AX', 'Intensity_UpperQuartileIntensity_Cycle02_CDK2', 'Intensity_UpperQuartileIntensity_Cycle02_DAPI', 'Intensity_UpperQuartileIntensity_Cycle02_DIC', 'Intensity_UpperQuartileIntensity_Cycle02_IRF3', 'Intensity_UpperQuartileIntensity_Cycle02_pATM(1981)', 'Intensity_UpperQuartileIntensity_Cycle03_ATM', 'Int

/tmp/ipykernel_2767378/2450547278.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pc_df['_label_str'] = pc_df[TARGET_LABEL].astype(str)
/tmp/ipykernel_2767378/2450547278.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pc_df['_label_id'] = label_encoder.fit_transform(pc_df['_label_str'])


### Optional: Explore Additional Features at Relaxed Missingness

If strict no-NaN mode leaves too few features, run the next cell to see which columns become available at 1%, 5%, 10%, and 20% missingness.

In [ ]:
thresholds = [0.0, 0.01, 0.05, 0.10, 0.20]
rows = []
for t in thresholds:
    cols_t = missing_frac[missing_frac <= t].index.tolist()
    rows.append({
        'max_missing_fraction': t,
        'n_features_available': len(cols_t),
        'first_15_features': cols_t[:15],
    })

relaxed_df = pd.DataFrame(rows)
relaxed_df

In [ ]:
point_clouds = []
labels_int = []
labels_str = []
cell_keys = []
num_points = []

group_cols = ['ImageNumber', 'ObjectNumber']
for (img, obj), g in pc_df.groupby(group_cols, sort=False):
    pts = g[point_feature_cols].to_numpy(dtype=np.float32)
    if pts.shape[0] < MIN_POINTS_PER_CELL:
        continue

    if pts.shape[0] > MAX_POINTS_PER_CELL:
        idx = np.random.choice(pts.shape[0], MAX_POINTS_PER_CELL, replace=False)
        pts = pts[idx]

    point_clouds.append(pts)
    labels_int.append(int(g['_label_id'].iloc[0]))
    labels_str.append(str(g['_label_str'].iloc[0]))
    cell_keys.append((int(img), int(obj)))
    num_points.append(int(pts.shape[0]))

print('Total cell point clouds:', len(point_clouds))
print('Min / median / max points per cloud:', int(np.min(num_points)), int(np.median(num_points)), int(np.max(num_points)))

coord_cols = ['Location_Center_X', 'Location_Center_Y', 'Location_Center_Z']
coord_in_selected = [c for c in coord_cols if c in point_feature_cols]
print('Coordinate columns used in point clouds:', coord_in_selected)

pd.Series(labels_str).value_counts().to_frame('count')

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x152b679e0290>>
Traceback (most recent call last):
  File "/home/sv496/project_pi_sk2433/sv496/HiPoNet/.venv/lib64/python3.11/site-packages/ipykernel/ipkernel.py", line 781, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


## 5) Save Reusable Artifacts (HiPoNet-ready + metadata)

In [ ]:
# Save point clouds as list of numpy arrays (variable length)
with open(OUT_DIR / 'pc_multifoci_cellstate.pkl', 'wb') as f:
    pickle.dump(point_clouds, f)

# Save integer labels
labels_np = np.array(labels_int, dtype=np.int64)
np.save(OUT_DIR / 'labels_multifoci_cellstate.npy', labels_np)

# Save human-readable class map
class_map = pd.DataFrame({
    'label_id': np.arange(len(label_encoder.classes_)),
    'label_name': label_encoder.classes_
})
class_map.to_csv(OUT_DIR / 'label_map.csv', index=False)

# Save per-sample metadata
meta_df = pd.DataFrame({
    'ImageNumber': [k[0] for k in cell_keys],
    'ObjectNumber': [k[1] for k in cell_keys],
    'label_id': labels_int,
    'label_name': labels_str,
    'n_points': num_points,
})
meta_df.to_csv(OUT_DIR / 'pointcloud_metadata.csv', index=False)

# Save torch-friendly version too
torch_point_clouds = [torch.tensor(pc, dtype=torch.float32) for pc in point_clouds]
torch.save(torch_point_clouds, OUT_DIR / 'pc_multifoci_cellstate.pt')
torch.save(torch.tensor(labels_int, dtype=torch.long), OUT_DIR / 'labels_multifoci_cellstate.pt')

print('Saved artifacts to', OUT_DIR)
print('Files:')
for p in sorted(OUT_DIR.glob('*')):
    print('-', p.name)

### Optional: Missingness Audit Only

This cell is just for inspection. The actual feature selection in the notebook uses **all numeric features with zero missing values**.

In [ ]:
thresholds = [0.0, 0.01, 0.05, 0.10, 0.20]
rows = []
for t in thresholds:
    cols_t = missing_frac[missing_frac <= t].index.tolist()
    rows.append({
        'max_missing_fraction': t,
        'n_features_available': len(cols_t),
        'first_15_features': cols_t[:15],
    })

relaxed_df = pd.DataFrame(rows)
relaxed_df